# [LAB-13] 웹 데이터 수집하기

## 2. HTML 데이터 처리(2)

### #01 준비작업

1. 라이브러리 참조하기

In [1]:
import requests
from bs4 import BeautifulSoup

### #02 데이터 요청하기

1. 웹 페이지의 모든 소스코드 가져오기

In [2]:
# 세션 객체 생성
with requests.Session() as session:
    # 세션 객체에 웹 브라우저 정보(UserAgent) 주입
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36 Edg/147.0.0.0"
    })

    url = "https://data.hossam.kr/py/sample.html"       # 요청할 데이터의 URL
    r = session.get(url)        # 요청 결과 받아오기 --> HTTP GET 요청

    # HTTP 상태값이 200이 아닌 경우는 강제로 에러를 발생시켜 코드의 진행을 중단시킴
    if r.status_code != 200:
        msg = "[%d Error] %s 에러가 발생함" % (r.status_code, r.reason)
        raise Exception(msg)

    print(r)

<Response [200]>


### #03 응답 결과 확인

1. 응답 결과 문자열 확인

In [3]:
# 수신된 데이터의 인코딩 설정 (한글이 깨진다면 eur-kr로 변경)
r.encoding = "utf-8"

# 수신된 결과 확인 --> 웹에서 가져온 모든 데이터는 문자열 형식임
print(type(r.text))
r.text

<class 'str'>


'<!DOCTYPE html><html lang="ko"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1.0"><title>Document</title><style>h1{ color: #f0f;} h2{ color: #06f;} .myclass{ color: #f00;} #myid{ color: #f60;} .syllabus >li >ol >li{ text-decoration: underline;} .syllabus ol{ font-weight: bold;} .part1{ background-color: #eeeeee;} .part2{ background-color: #d5d5d5;} div.sub.part1{ border: 1px dotted #000;} div.sub.part2#hello{ border: 1px solid #555;} a[href]{ font-size: 20px;} a[href=\'#\']{ color: green;} </style></head><body><h1>Hello World</h1><a>link0</a><a href="#">link1</a><a href="https://www.naver.com">link2</a><h2 id="myid">Python</h2><div class="sub part1"><ul class="syllabus"><li>변수와 데이터 타입</li><li class="myclass">연산자</li><li>연속성 자료형 <ol><li>리스트(list)</li><li>딕셔너리(dict)</li><li>집합(set)</li></ol></li><li>프로그램 흐름제어</li><li>함수</li></ul></div><h2>Data Analysis</h2><div class="sub part2" id="hello"><ul><li>데이터 수집</li><li class="myclass">데이터 전처리</li>

2. 응답 결과 문자열을 BeautifelSoup 객체로 변환

In [4]:
soup = BeautifulSoup(r.text)
print(type(soup))
soup

<class 'bs4.BeautifulSoup'>


<!DOCTYPE html>
<html lang="ko"><head><meta charset="utf-8"/><meta content="width=device-width, initial-scale=1.0" name="viewport"/><title>Document</title><style>h1{ color: #f0f;} h2{ color: #06f;} .myclass{ color: #f00;} #myid{ color: #f60;} .syllabus >li >ol >li{ text-decoration: underline;} .syllabus ol{ font-weight: bold;} .part1{ background-color: #eeeeee;} .part2{ background-color: #d5d5d5;} div.sub.part1{ border: 1px dotted #000;} div.sub.part2#hello{ border: 1px solid #555;} a[href]{ font-size: 20px;} a[href='#']{ color: green;} </style></head><body><h1>Hello World</h1><a>link0</a><a href="#">link1</a><a href="https://www.naver.com">link2</a><h2 id="myid">Python</h2><div class="sub part1"><ul class="syllabus"><li>변수와 데이터 타입</li><li class="myclass">연산자</li><li>연속성 자료형 <ol><li>리스트(list)</li><li>딕셔너리(dict)</li><li>집합(set)</li></ol></li><li>프로그램 흐름제어</li><li>함수</li></ul></div><h2>Data Analysis</h2><div class="sub part2" id="hello"><ul><li>데이터 수집</li><li class="myclass">데이터 전처리</li>

### #04 HTML 태그에 의한 추출

1. `<h1>` 태그를 갖는 요소에 접근한다.

In [5]:
myselect = soup.select("h1")
print(type(myselect))
myselect

<class 'bs4.element.ResultSet'>


[<h1>Hello World</h1>]

2. HTML 태그에 의한 추출

In [6]:
mytag = myselect[0]
print(type(mytag))
mytag

<class 'bs4.element.Tag'>


<h1>Hello World</h1>

3. 추출한 태그에서 텍스트만 추출

In [7]:
mytext = mytag.text.strip()
mytext

'Hello World'

### #05 class에 의한 데이터 추출

1. 복수 요소에게 적용되는 속성

In [8]:
myselect = soup.select(".myclass")
myselect

[<li class="myclass">연산자</li>,
 <li class="myclass">데이터 전처리</li>,
 <ol class="myclass"><li>기초통계</li><li>데이터 시각화</li></ol>]

2. 복수 요소이므로 반복문을 통해 처리

In [9]:
for i, v in enumerate(myselect):
    # 추출한 요소가 하위 태그를 포함하는 경우 그 안의 텍스트만 일괄 추출
    print("%d번째 요소 : %s" % (i, v.text.strip()))

0번째 요소 : 연산자
1번째 요소 : 데이터 전처리
2번째 요소 : 기초통계데이터 시각화


3. 특정 HTML 태그 객체의 하위 요소 추출하기

In [12]:
myli = myselect[2].select("li")
myli

[<li>기초통계</li>, <li>데이터 시각화</li>]

In [13]:
for i in myli:
    print(i.text.strip())

기초통계
데이터 시각화


### #06 id에 의한 추출

1. id 속성은 페이지 내에서 고유한 요소임을 의미

In [14]:
myselect = soup.select("#myid")
myselect

[<h2 id="myid">Python</h2>]

In [15]:
print(myselect[0].text.strip())             # .text : 태그를 제외하고 사람이 보는 글자 부분만 가져옴

Python


### #07 여러 요소에 동시 접근하기

1. 다양한 선택자를 콤마(,)로 구분하여 지정

In [16]:
items = soup.select("#myid, .myclass")
items

[<h2 id="myid">Python</h2>,
 <li class="myclass">연산자</li>,
 <li class="myclass">데이터 전처리</li>,
 <ol class="myclass"><li>기초통계</li><li>데이터 시각화</li></ol>]

In [17]:
for i in items:
    print(i.text)

Python
연산자
데이터 전처리
기초통계데이터 시각화


### #08 복합 선택자

1. 자식 선택자

In [18]:
soup.select(".syllabus > .myclass")

[<li class="myclass">연산자</li>]

2. 자손 선택자

In [19]:
soup.select(".part1 .myclass")

[<li class="myclass">연산자</li>]

3. 속성 선택자

In [21]:
myselect = soup.select("a[href]")
myselect

[<a href="#">link1</a>, <a href="https://www.naver.com">link2</a>]

4. href 속성에 적용되어 있는 값 추출하기

In [22]:
# 속성값은 각 태그요소의 attrs 프로퍼티로 접근 가능 --> dict 형태
for i, v in enumerate(myselect):
    print("----[%d]----" % i)
    print(v.attrs)

    # 딕셔너리에 대한 in 연산자는 key의 존재 여부를 판별
    if "href" in v.attrs:
        print("%d번째의 href속성값 : %s" % (i, v.attrs["href"]))

----[0]----
{'href': '#'}
0번째의 href속성값 : #
----[1]----
{'href': 'https://www.naver.com'}
1번째의 href속성값 : https://www.naver.com
